# Sesión 10 A



## 2.2. Patrones de razonamiento e inferencia

Teniendo una situación modelada con una red Bayesiana, nos podemos plantear **tres** tipos básicos de razonamiento de podríamos querer resolver:

* Razonamiento causal
* Razonamiento evidencial
* Razonamiento intercausal

**1. Razonamiento causal**

El **razonamiento causal** sigue la dirección natural de las flechas del grafo: va de **causa → efecto**, o de **nodo padre → nodo hijo**.

> Si sé algo sobre las causas, ¿qué puedo inferir sobre sus efectos?

![causal-reasoning](../images/sesion10-student-model-causal.png)

Por ejemplo, 

**Pregunta**: ¿cuál es la probabilidad de obtener una buena carta de recomendación?

$$P(r^1) = \sum_{D,I,C,E} P(D,I,C,E,r^1) \approx ?$$

In [1]:
from pgmpy.models import BayesianNetwork, DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD

In [2]:
student_model = DiscreteBayesianNetwork(
    [("D", "C"), ("I", "C"), ("I", "E"), ("C", "R")]
)

# CPDs
cpd_D = TabularCPD(
    variable='D',
    variable_card=2,
    values=[
        [0.6],
        [0.4]
    ]
)
cpd_I = TabularCPD(
    variable='I',
    variable_card=2,
    values=[
        [0.7],
        [0.3]
    ]
)

cpd_C = TabularCPD(
    variable='C',
    variable_card=3,
    values=[
        [0.30, 0.70, 0.02, 0.20],
        [0.40, 0.25, 0.08, 0.30],
        [0.30, 0.05, 0.90, 0.50]
    ],
    evidence=['I', 'D'], 
    evidence_card=[2, 2] 
)
cpd_E = TabularCPD(
    variable='E',
    variable_card=2,
    values=[
        [0.95, 0.20],
        [0.05, 0.80]
    ],
    evidence=['I'],
    evidence_card=[2]
)
cpd_R = TabularCPD(
    variable='R',
    variable_card=2,
    values=[
        [0.99, 0.40, 0.10],
        [0.01, 0.60, 0.90]
    ],
    evidence=['C'],
    evidence_card=[3]
)

In [3]:
student_model.add_cpds(cpd_D, cpd_I, cpd_C, cpd_E, cpd_R)

In [4]:
# Obtenemos la distribución conjunta de la red


In [5]:
#print

In [6]:
# Marginalizar sobre las variables I, D, C, E


In [25]:
#reduce 

Sin embargo, podemos evaluar cómo esta probabilidad cambia si la condicionamos sobre la inteligencia. Por ejemplo, si el estudiante no es muy inteligente

$$P(r^1 | i^0) = \frac{P(r^1, i^0)}{P(i^0)} = \frac{\sum_{D,C,E} P(D, i^0, C, E, r^1)}{\sum_{D,C,E,R} P(D, i^0, C, E, R)} \approx ?$$

In [8]:
# marginalizar sobre las variables D, C, E // numerador

In [9]:
# get_value


In [10]:
# marginalizar sobre las variables D, C, E, R // denominador

In [11]:
# get_value

**¿se esperaba esto o no?**

In [12]:
# print probabilidad de r1 dado i0


Por otra parte, si también condicionamos sobre la dificultad

$$P(r^1 | i^0, d^0) = \frac{P(r^1, i^0, d^0)}{P(i^0, d^0)} = \frac{\sum_{C,P} P(d^0, i^0, C, E, r^1)}{\sum_{C,P,R} P(d^0, i^0, C, E, R)} \approx ?$$

In [13]:
# Marginalizar sobre las variables C, E, R // numerador

In [14]:
# Marginalizar sobre las variables C, E, R, I // denominador

**¿Se esperaba esto o no?**

In [15]:
#print probabilidad de r1 dado i0 y d0


---

**2. Razonamiento evidencial**

Va **de efecto a causa**, en sentido contrario a las flechas.

> Si observo un efecto, ¿qué puedo inferir sobre sus causas?

![causal-reasoning](../images/sesion10-student-model-evid.png)

Por ejemplo, la probabilidad de que el curso sea difícil es:

$$P(d^1) = 0.4$$

Condicionando sobre la calificación:

$$P(d^1 | c^0) = \frac{P(d^1, c^0)}{P(c^0)} = \frac{\sum_{I,E,R} P(d^1, I, c^0, E, R)}{\sum_{D,I,E,R} P(D, I, c^0, E, R)} \approx?$$

In [16]:
# marginalizar sobre las variables I, E, R // numerador

In [17]:
# marginalizar sobre las variables D, I, E, R // denominador

In [18]:
# print probabilidad de d1 dado c0

In [19]:
#Otra forma de calcular P(D1 | C0) usando inferencia en la red bayesiana    
from pgmpy.inference import VariableElimination


> Intuición: observar una calificación baja hace más probable que el curso haya sido difícil (sube de $0.4$ a $\approx 0.63$).

---

Similarmente, la probabilidad de que el estudiante sea inteligente es:

$$P(i^1) = 0.3$$

Condicionando sobre la calificación:

$$P(i^1 | c^0) = \frac{P(i^1, c^0)}{P(c^0)} = \frac{\sum_{D,E,R} P(D, i^1, c^0, E, R)}{\sum_{D,I,E,R} P(D, I, c^0, E, R)} \approx ?$$

In [20]:
# marginalizar sobre las variables D, E, R // numerador


# marginalizar sobre las variables D, I, E, R // denominador


In [21]:
# print probabilidad de i1 dado c0

In [22]:
# o con VariableElimination


> Intuición: observar una calificación baja hace menos probable que el estudiante sea inteligente (baja del $0.3$ a $\approx 0.11$).

**3. Razonamiento intercausal**

Ocurre cuando **dos causas comparten un mismo efecto** y una de ellas se observa.

> Si conozco una causa, ¿cómo cambia mi creencia sobre la otra, dado que comparten el mismo efecto?


![intercausal-reasoning](../images/sesion10-student-model-inter.png )

$\text{Dificultad} \longrightarrow \text{Calificación} \longleftarrow \text{Inteligencia}$

Normalmente, $D$ e $I$ son independientes. Pero, una vez que conocemos el efecto común -por ejemplo, la calificación $C$-, dejan de serlo.

> Si sabemos que la calificación fue alta y que el curso era difícil, es más probable que el estudiante haya sido inteligente.

Antes de observar $C$:

$$ D \perp I $$

Después de observar $C$:
$$ D \not\perp I \mid C $$

De nuevo, la probabilidad de que el estudiante sea inteligente es:

$$P(i^1) = 0.3$$

Condicionando sobre la calificación:

$$P(i^1 | c^0) = \frac{P(i^1, c^0)}{P(c^0)} \approx 0.07$$

Aún más, si condicionamos sobre la dificultad:

$$P(i^1 | c^0, d^1) = \frac{P(i^1, c^0, d^1)}{P(c^0, d^1)} \approx ?$$

In [23]:
# VariableElimination

> Inicialmente, el estudiante tiene una probabilidad moderada de ser inteligente ($P(i^1)=0.3$). Al observar que obtuvo una **mala calificación**, esa creencia **disminuye drásticamente** ($P(i^1 \mid c^0) \approx 0.07$). Sin embargo, si además sabemos que el curso era **difícil**, parte de la mala nota se explica por la dificultad, por lo que la probabilidad de que sea inteligente **vuelve a subir ligeramente** $P(i^1 \mid c^0, d^1) \approx 0.11$.

In [24]:
#guardar el modelo
#import pickle

#with open('student-model.pkl', 'wb') as f:
#    pickle.dump(student_model, f)